# 07 · Análise: respondendo às perguntas de negócio

**Problema:** quais problemas estão ganhando peso nas reclamações contra bancos e instituições de pagamento entre 2021 e 2026, e esse crescimento vem acompanhado de piora na resolução?

Todas as consultas leem só a camada Gold (`mvp_reclamacoes.gold`).

**Como rodar**
- **No VS Code:** cada seção imprime os números com `print`. É a fonte da discussão no README.
- **No workspace:** o `display()` de cada seção vira gráfico pelo editor de visualização, seguindo a especificação escrita na própria seção. No kernel local, o `display()` só mostra `DataFrame[...]`.

**Janelas de comparação**
- **Base:** jan–dez/2022.
- **Fim:** set/2025–ago/2026.

A base de 2022 evita a reorganização da taxonomia de dados pessoais feita em 2021-09/10 (notebook 03, H2b). Janelas de 12 meses anulam a sazonalidade.

## 0. Parâmetros e funções de apoio

In [ ]:
from pyspark.sql import functions as F

GOLD = "mvp_reclamacoes.gold"
JANELA_BASE = ("2022-01", "2022-12")
JANELA_FIM = ("2025-09", "2026-08")


def mostrar(df, n=100):
    """Imprime até n linhas de um DataFrame pequeno, com as colunas alinhadas."""
    linhas = df.limit(n + 1).collect()
    colunas = df.columns
    textos = [[("NULL" if v is None else str(v)) for v in linha] for linha in linhas[:n]]
    larguras = [max([len(c)] + [len(t[i]) for t in textos]) for i, c in enumerate(colunas)]
    print("  ".join(c.ljust(w) for c, w in zip(colunas, larguras)))
    for t in textos:
        print("  ".join(v.rjust(w) for v, w in zip(t, larguras)))
    if len(linhas) > n:
        print(f"... mais de {n} linhas, mostrando só as {n} primeiras")

## P1. Como evoluiu o volume de reclamações do recorte bancário, e qual a sua participação no total da plataforma?

**Por que a pergunta existe:** o uso do consumidor.gov.br cresce ao longo do período. Por isso, o crescimento do setor precisa ser lido junto com o da plataforma inteira. É também por esse motivo que as perguntas seguintes usam **participação** em vez de volume.

**Fonte:** `gold.agg_reclamacoes_mensais` (`total_recorte` e `total_plataforma` por mês).

**Gráficos no workspace**

| Gráfico | Tipo | Eixo X | Eixo Y |
|---|---|---|---|
| 1 | Linha | `ano_mes` | `total_recorte` e `total_plataforma` |
| 2 | Linha | `ano_mes` | `participacao_pct` |

In [ ]:
mensal = (
    spark.table(f"{GOLD}.agg_reclamacoes_mensais")
    .withColumn("participacao_pct", F.round(100 * F.col("total_recorte") / F.col("total_plataforma"), 2))
    .orderBy("ano_mes")
)
mostrar(mensal, 70)
display(mensal)

### P1 por ano e nas janelas de comparação

O ano de 2026 tem só 8 meses (jan–ago), por isso a média mensal é a medida comparável entre os anos.

In [ ]:
anual = (
    mensal.groupBy(F.substring("ano_mes", 1, 4).alias("ano"))
    .agg(F.count("*").alias("meses"), F.sum("total_recorte").alias("recorte"), F.sum("total_plataforma").alias("plataforma"))
    .withColumn("media_mensal_recorte", F.round(F.col("recorte") / F.col("meses")).cast("long"))
    .withColumn("media_mensal_plataforma", F.round(F.col("plataforma") / F.col("meses")).cast("long"))
    .withColumn("participacao_pct", F.round(100 * F.col("recorte") / F.col("plataforma"), 2))
    .orderBy("ano")
)
mostrar(anual)
display(anual)

janela = (
    F.when(F.col("ano_mes").between(*JANELA_BASE), "base 2022")
    .when(F.col("ano_mes").between(*JANELA_FIM), "fim (set/2025-ago/2026)")
)
janelas = (
    mensal.withColumn("janela", janela).filter("janela IS NOT NULL")
    .groupBy("janela")
    .agg(F.count("*").alias("meses"), F.sum("total_recorte").alias("recorte"), F.sum("total_plataforma").alias("plataforma"))
    .withColumn("participacao_pct", F.round(100 * F.col("recorte") / F.col("plataforma"), 2))
    .orderBy("janela")
)
print()
mostrar(janelas)
por_janela = {r["janela"].split()[0]: r for r in janelas.collect()}  # "base" e "fim"
base, fim = por_janela["base"], por_janela["fim"]
print(f"\nrecorte: {fim['recorte'] / base['recorte']:.2f}x o volume da base | "
      f"plataforma: {fim['plataforma'] / base['plataforma']:.2f}x | "
      f"participação: {base['participacao_pct']}% → {fim['participacao_pct']}% "
      f"({fim['participacao_pct'] - base['participacao_pct']:+.2f} p.p.)")

## P2. Quais grupos de problema e quais problemas mais ganharam e mais perderam participação?

**Métrica:** a participação de cada grupo ou problema no total de reclamações do recorte em cada janela, e a variação em pontos percentuais (p.p.) entre a base (2022) e o fim (set/2025–ago/2026). A variação é calculada com as participações sem arredondamento.

**Sensibilidade:** a coluna `variacao_pp_sem_repetidas` repete o cálculo **sem** as linhas marcadas como `linha_repetida` (possíveis duplicatas; notebook 03, H9). Se as duas variações forem parecidas, as repetidas não mudam a conclusão.

**Fonte:** `gold.fato_reclamacao` + `gold.dim_tempo` (`ano_mes`) + `gold.dim_problema`.

**Gráficos no workspace**

| Gráfico | Tipo | Eixo X | Eixo Y |
|---|---|---|---|
| 1 | Barras | `grupo_problema` | `base_pct` e `fim_pct` (barras agrupadas) |
| 2 | Barras horizontais | `problema` (os 15 maiores ganhos) | `variacao_pp` |

In [ ]:
fato = spark.table(f"{GOLD}.fato_reclamacao")
tempo = spark.table(f"{GOLD}.dim_tempo").select("sk_tempo", "ano_mes")
problema = spark.table(f"{GOLD}.dim_problema")
janela_mes = F.when(F.col("ano_mes").between(*JANELA_BASE), "base").when(F.col("ano_mes").between(*JANELA_FIM), "fim")
fato_janelas = (
    fato.join(tempo, "sk_tempo")
    .withColumn("janela", janela_mes).filter("janela IS NOT NULL")
    .join(problema, "sk_problema")
)


def variacao_participacao(df, colunas):
    """Participação (%) de cada valor de `colunas` no total da janela, na base e no fim, e a variação em p.p."""
    totais = df.groupBy().pivot("janela", ["base", "fim"]).count().first()
    contagem = df.groupBy(*colunas).pivot("janela", ["base", "fim"]).count().fillna(0)
    return (
        contagem
        .withColumn("base_pct", F.round(100 * F.col("base") / totais["base"], 2))
        .withColumn("fim_pct", F.round(100 * F.col("fim") / totais["fim"], 2))
        .withColumn("variacao_pp", F.round(100 * (F.col("fim") / totais["fim"] - F.col("base") / totais["base"]), 2))
    )


def com_sensibilidade(df, colunas):
    """Acrescenta a variação calculada sem as linhas repetidas."""
    sem_repetidas = variacao_participacao(df.filter(~F.col("linha_repetida")), colunas).select(
        *colunas, F.col("variacao_pp").alias("variacao_pp_sem_repetidas"))
    return variacao_participacao(df, colunas).join(sem_repetidas, colunas, "left")


totais = fato_janelas.groupBy("janela").count().orderBy("janela").collect()
print("reclamações por janela: " + " | ".join(f"{r['janela']}: {r['count']:,}" for r in totais))

### P2 por grupo de problema

In [ ]:
grupos = com_sensibilidade(fato_janelas, ["grupo_problema"]).orderBy(F.desc("variacao_pp"))
mostrar(grupos)
display(grupos)

### P2 por problema: maiores ganhos e maiores perdas de participação

In [ ]:
problemas = com_sensibilidade(fato_janelas, ["grupo_problema", "problema"])
print("== 15 maiores ganhos de participação")
mostrar(problemas.orderBy(F.desc("variacao_pp")).select("problema", "grupo_problema", "base", "fim", "base_pct", "fim_pct",
                                                        "variacao_pp", "variacao_pp_sem_repetidas"), 15)
print("\n== 15 maiores perdas de participação")
mostrar(problemas.orderBy("variacao_pp").select("problema", "grupo_problema", "base", "fim", "base_pct", "fim_pct",
                                                "variacao_pp", "variacao_pp_sem_repetidas"), 15)
print(f"\nproblemas com registros só no fim (temas novos): {problemas.filter('base = 0').count()} | "
      f"só na base: {problemas.filter('fim = 0').count()} | total: {problemas.count()}")
display(problemas.orderBy(F.desc("variacao_pp")).limit(15))

## Problemas em alta

São os 3 problemas com maior ganho de participação na P2. Há um salto claro entre o 3º colocado (+4,69 p.p.) e o 4º (+1,89 p.p.). A lista é lida do resultado da P2, e não digitada à mão.

In [ ]:
EM_ALTA = [linha["problema"] for linha in problemas.orderBy(F.desc("variacao_pp")).limit(3).collect()]
assert len(EM_ALTA) == 3
for posicao, nome in enumerate(EM_ALTA, 1):
    print(f"{posicao}. {nome}")

### P2b. O ganho foi gradual ou abrupto?

Mostra a participação mensal de cada problema em alta no recorte, de jan/2021 a ago/2026.
- **Subida gradual:** sugere uma tendência real.
- **Salto de um mês para o outro:** pode indicar uma mudança na forma de classificar as reclamações. Nesse caso, a leitura da P2 precisa de cautela.

**Gráfico no workspace:** linha; eixo X = `ano_mes`; eixo Y = participação; uma série por problema.

In [ ]:
por_mes = fato.join(tempo, "sk_tempo").join(problema, "sk_problema")
total_mes = por_mes.groupBy("ano_mes").agg(F.count("*").alias("total"))
serie = (
    por_mes.filter(F.col("problema").isin(EM_ALTA))
    .groupBy("ano_mes").pivot("problema", EM_ALTA).count().fillna(0)
    .join(total_mes, "ano_mes")
)
for i, nome in enumerate(EM_ALTA, 1):
    serie = serie.withColumn(f"p{i}_pct", F.round(100 * F.col(f"`{nome}`") / F.col("total"), 2))
serie = serie.select("ano_mes", "total", *[f"p{i}_pct" for i in range(1, 4)]).orderBy("ano_mes")
print("p1, p2 e p3 = participação (%) de cada problema em alta, na ordem da lista acima")
mostrar(serie, 70)
display(serie)

## P3. Os problemas em alta são resolvidos pior que o restante do recorte? A resolução deles piorou?

**Métricas**, calculadas por categoria e janela:

| Métrica | Definição |
|---|---|
| `taxa_resposta_pct` | Reclamações respondidas pela empresa |
| `avaliadas_pct` | Reclamações avaliadas pelo consumidor |
| `indice_oficial_pct` | Resolvidas + não avaliadas, pela definição oficial do consumidor.gov.br. A não avaliada conta como resolvida |
| `taxa_entre_avaliadas_pct` | Resolvidas entre as avaliadas com resultado registrado |
| `satisfacao_media` | Nota média de 1 a 5, só entre as avaliadas |
| `prazo_medio_dias` | Tempo médio de resposta, só entre as respondidas com tempo válido |

As categorias são cada um dos 3 problemas em alta, mais "Demais problemas" e "Recorte (todos)", na base (2022) e no fim (set/2025–ago/2026).

**Gráfico no workspace:** barras agrupadas; eixo X = `categoria`; eixo Y = `taxa_entre_avaliadas_pct`; uma série por `janela`. Repetir com `indice_oficial_pct`.

In [ ]:
def metricas(df):
    return df.agg(
        F.count("*").alias("registros"),
        F.round(100 * F.avg(F.col("foi_respondida").cast("double")), 2).alias("taxa_resposta_pct"),
        F.round(100 * F.avg(F.col("foi_avaliada").cast("double")), 2).alias("avaliadas_pct"),
        F.round(100 * F.avg((F.col("foi_resolvida") | ~F.col("foi_avaliada")).cast("double")), 2).alias("indice_oficial_pct"),
        F.round(100 * F.avg(F.col("foi_resolvida").cast("double")), 2).alias("taxa_entre_avaliadas_pct"),
        F.round(F.avg("nota_consumidor"), 2).alias("satisfacao_media"),
        F.round(F.avg("tempo_resposta_dias"), 2).alias("prazo_medio_dias"),
    )


categoria = F.when(F.col("problema").isin(EM_ALTA), F.col("problema")).otherwise(F.lit("Demais problemas"))
por_categoria = metricas(fato_janelas.withColumn("categoria", categoria).groupBy("categoria", "janela"))
recorte_todo = metricas(fato_janelas.groupBy("janela")).withColumn("categoria", F.lit("Recorte (todos)"))
ordem = {nome: i for i, nome in enumerate(EM_ALTA + ["Demais problemas", "Recorte (todos)"])}
p3 = por_categoria.unionByName(recorte_todo)
linhas = sorted(p3.collect(), key=lambda r: (ordem[r["categoria"]], r["janela"]))
p3 = spark.createDataFrame(linhas, p3.schema)
mostrar(p3.select("categoria", "janela", "registros", "taxa_resposta_pct", "avaliadas_pct", "indice_oficial_pct",
                  "taxa_entre_avaliadas_pct", "satisfacao_media", "prazo_medio_dias"))
display(p3)